In [13]:
import pandas as pd

In [14]:
df = pd.read_csv("mp_sponsorwise_dataset.csv")
df.head()

,event_id,brand_id,state,city,event_type,month,day_of_week,is_weekend,is_festive,festival_name,...,brand_city_focus,brand_annual_budget,brand_activation_maturity,fit_score,sponsor_amount,activation_quality,clutter_index,brand_lift,roi,feasible_to_sponsor
0,1,685,Madhya Pradesh,Ujjain,Food Festival,3,0,0,1,festive,...,metro,1193012,0.746,0.3041,114490,0.9700,0.2792,2.143,-0.6719,0
1,1,609,Madhya Pradesh,Ujjain,Food Festival,3,0,0,1,festive,...,metro,2022412,0.395,0.2500,55563,0.8158,0.3478,2.390,-0.4607,0
2,1,217,Madhya Pradesh,Ujjain,Food Festival,3,0,0,1,festive,...,all_mp,3073226,0.422,0.9975,167307,0.8554,0.3350,4.823,-0.9759,0
3,1,343,Madhya Pradesh,Ujjain,Food Festival,3,0,0,1,festive,...,all_mp,1195718,0.786,0.7790,143486,0.9700,0.1625,4.692,1.2899,1
4,1,755,Madhya Pradesh,Ujjain,Food Festival,3,0,0,1,festive,...,tier2,1540216,0.596,0.3999,144718,0.9637,0.1752,3.719,-0.6718,0


In [15]:
cols_to_drop = ['event_id', 'brand_id', 'state', 'festival_name']

In [16]:
leakage_cols = [
    'actual_attendance_rate', 
    'event_rating', 
    'event_success', 
    'activation_quality', 
    'clutter_index', 
    'brand_lift', 
    'roi',
    'predicted_attendance',       
    'predicted_attendance_rate',  
    'predicted_impressions'       
]

In [17]:
df_clean = df.drop(columns=cols_to_drop + leakage_cols)
df_clean.head(10)

,city,event_type,month,day_of_week,is_weekend,is_festive,is_indoor,temperature,is_raining,humidity,...,competing_events,actual_attendance,brand_category,brand_kpi,brand_city_focus,brand_annual_budget,brand_activation_maturity,fit_score,sponsor_amount,feasible_to_sponsor
0,Ujjain,Food Festival,3,0,0,1,0,28.31,0,25.91,...,1,6865,Real Estate,hybrid,metro,1193012,0.746,0.3041,114490,0
1,Ujjain,Food Festival,3,0,0,1,0,28.31,0,25.91,...,1,6865,Fintech,hybrid,metro,2022412,0.395,0.2500,55563,0
2,Ujjain,Food Festival,3,0,0,1,0,28.31,0,25.91,...,1,6865,FMCG,awareness,all_mp,3073226,0.422,0.9975,167307,0
3,Ujjain,Food Festival,3,0,0,1,0,28.31,0,25.91,...,1,6865,Beauty/Personal Care,leads,all_mp,1195718,0.786,0.7790,143486,1
4,Ujjain,Food Festival,3,0,0,1,0,28.31,0,25.91,...,1,6865,Automobile,hybrid,tier2,1540216,0.596,0.3999,144718,0
5,Ujjain,Food Festival,3,0,0,1,0,28.31,0,25.91,...,1,6865,FMCG,leads,all_mp,1509975,0.378,0.9975,124370,1
6,Ujjain,Food Festival,3,0,0,1,0,28.31,0,25.91,...,1,6865,Telecom,awareness,metro,4030086,0.700,0.5529,108285,0
7,Burhanpur,College Fest,4,2,0,0,0,33.60,0,34.52,...,0,8317,Local Retail,hybrid,tier2,890714,0.497,0.2500,106885,0
8,Burhanpur,College Fest,4,2,0,0,0,33.60,0,34.52,...,0,8317,Real Estate,leads,all_mp,4265812,0.665,0.2612,207120,1
9,Burhanpur,College Fest,4,2,0,0,0,33.60,0,34.52,...,0,8317,Apparel,leads,all_mp,2023028,0.636,0.7790,242763,0


In [18]:
categorical_cols = [
    'city', 
    'event_type', 
    'brand_category', 
    'brand_kpi', 
    'brand_city_focus'
]

In [19]:
df_features = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

In [20]:
bool_cols = df_features.select_dtypes(include=['bool']).columns
df_features[bool_cols] = df_features[bool_cols].astype(int)


In [21]:
target_1 = 'actual_attendance'
target_2 = 'feasible_to_sponsor'

In [22]:
feature_cols = [c for c in df_features.columns if c not in [target_1, target_2]]
final_cols = feature_cols + [target_1, target_2]
df_features = df_features[final_cols]

In [23]:
df_features.head()

,month,day_of_week,is_weekend,is_festive,is_indoor,temperature,is_raining,humidity,venue_capacity,ticket_price,...,brand_category_Real Estate,brand_category_Telecom,brand_kpi_hybrid,brand_kpi_leads,brand_kpi_sales,brand_city_focus_metro,brand_city_focus_pilgrimage,brand_city_focus_tier2,actual_attendance,feasible_to_sponsor
0,3,0,0,1,0,28.31,0,25.91,6865,200,...,1,0,1,0,0,1,0,0,6865,0
1,3,0,0,1,0,28.31,0,25.91,6865,200,...,0,0,1,0,0,1,0,0,6865,0
2,3,0,0,1,0,28.31,0,25.91,6865,200,...,0,0,0,0,0,0,0,0,6865,0
3,3,0,0,1,0,28.31,0,25.91,6865,200,...,0,0,0,1,0,0,0,0,6865,1
4,3,0,0,1,0,28.31,0,25.91,6865,200,...,0,0,1,0,0,0,0,1,6865,0


In [24]:
df_features.to_csv("mp_sponsorwise_ml_features.csv", index=False)